# Questão 6 — Dimensão de Calendário

### O problema do estagiário

Ao fazer `GROUP BY dia_semana` direto na tabela de vendas, dias sem venda são ignorados — eles simplesmente não existem na tabela. Isso infla a média, pois o denominador conta só os dias em que houve venda, não todos os dias em que a loja esteve aberta.

Solução: construir uma dimensão de datas com todos os dias do período e fazer um `LEFT JOIN` com as vendas. Dias sem venda recebem `valor = 0` e entram no cálculo da média corretamente.

---

In [1]:
import sqlite3
import pandas as pd

# Carrega vendas e corrige datas
df_vendas = pd.read_csv('../data/raw/vendas_2023_2024.csv')

def parse_date(s):
    for fmt in ('%Y-%m-%d', '%d-%m-%Y'):
        try: return pd.to_datetime(s, format=fmt)
        except: pass

df_vendas['sale_date'] = df_vendas['sale_date'].apply(parse_date)

# Conecta no banco
conn = sqlite3.connect('../database/lighthouse.db')
df_vendas.to_sql('vendas', conn, if_exists='replace', index=False)

data_min = df_vendas['sale_date'].min().date()
data_max = df_vendas['sale_date'].max().date()
dias_com_venda = df_vendas['sale_date'].dt.date.nunique()
total_dias = (df_vendas['sale_date'].max() - df_vendas['sale_date'].min()).days + 1

print(f'Período: {data_min} → {data_max}')
print(f'Total de dias no período: {total_dias}')
print(f'Dias com venda: {dias_com_venda}')
print(f'Dias sem venda (seriam ignorados pelo estagiário): {total_dias - dias_com_venda}')

Período: 2023-01-01 → 2024-12-31
Total de dias no período: 731
Dias com venda: 725
Dias sem venda (seriam ignorados pelo estagiário): 6


Construindo a dimensão de datas no SQLite

O SQLite não tem uma função nativa de geração de sequência de datas como o PostgreSQL (`generate_series`). A solução é usar uma tabela auxiliar de números com CTE recursiva para gerar todos os dias do período.

A CTE `dim_calendario` gera uma linha para cada dia entre a data mínima e máxima das vendas, e já calcula:
- `data` — a data do dia
- `dia_semana_num` — número do dia (0 = Domingo no SQLite com `strftime('%w')`)
- `dia_semana` — nome em português

In [2]:
# Cria a tabela dim_calendario diretamente no banco
conn.execute('DROP TABLE IF EXISTS dim_calendario')

conn.execute("""
CREATE TABLE dim_calendario AS

-- CTE recursiva: gera uma linha para cada dia do período
WITH RECURSIVE sequencia_datas(data) AS (

    -- Ponto de partida: menor data da tabela de vendas
    SELECT DATE(MIN(sale_date))
    FROM vendas

    UNION ALL

    -- Incrementa um dia por vez até chegar na data máxima
    SELECT DATE(data, '+1 day')
    FROM sequencia_datas
    WHERE data < (SELECT DATE(MAX(sale_date)) FROM vendas)
)

SELECT
    data,

    -- strftime('%w') retorna 0=Domingo, 1=Segunda, ..., 6=Sábado
    CAST(strftime('%w', data) AS INTEGER)   AS dia_semana_num,

    CASE strftime('%w', data)
        WHEN '0' THEN 'Domingo'
        WHEN '1' THEN 'Segunda-feira'
        WHEN '2' THEN 'Terça-feira'
        WHEN '3' THEN 'Quarta-feira'
        WHEN '4' THEN 'Quinta-feira'
        WHEN '5' THEN 'Sexta-feira'
        WHEN '6' THEN 'Sábado'
    END                                     AS dia_semana

FROM sequencia_datas;
""")

n = conn.execute('SELECT COUNT(*) FROM dim_calendario').fetchone()[0]
print(f'Dimensão de calendário criada: {n} dias')

pd.read_sql('SELECT * FROM dim_calendario LIMIT 10', conn)

Dimensão de calendário criada: 731 dias


,data,dia_semana_num,dia_semana
0,2023-01-01,0,Domingo
1,2023-01-02,1,Segunda-feira
2,2023-01-03,2,Terça-feira
3,2023-01-04,3,Quarta-feira
4,2023-01-05,4,Quinta-feira
5,2023-01-06,5,Sexta-feira
6,2023-01-07,6,Sábado
7,2023-01-08,0,Domingo
8,2023-01-09,1,Segunda-feira
9,2023-01-10,2,Terça-feira


Cruzar calendário com vendas e calcular a média correta

O `LEFT JOIN` garante que todos os dias do calendário apareçam, mesmo sem venda.
O `COALESCE(..., 0)` substitui o `NULL` dos dias sem venda por `0`.

In [3]:
query = """
-- Passo 1: soma as vendas por dia (dias sem venda ficam NULL no LEFT JOIN)
WITH vendas_diarias AS (
    SELECT
        DATE(sale_date)    AS data,
        SUM(total)         AS total_dia
    FROM vendas
    GROUP BY DATE(sale_date)
)

-- Passo 2: cruza o calendário com as vendas e calcula a média por dia da semana
SELECT
    cal.dia_semana_num,
    cal.dia_semana,
    COUNT(cal.data)                              AS total_dias,
    SUM(COALESCE(v.total_dia, 0))                AS total_vendas,
    ROUND(AVG(COALESCE(v.total_dia, 0)), 2)      AS media_vendas

FROM dim_calendario cal

-- LEFT JOIN: dias sem venda ficam com NULL, transformado em 0 pelo COALESCE
LEFT JOIN vendas_diarias v ON cal.data = v.data

GROUP BY cal.dia_semana_num, cal.dia_semana

-- Ordena de Segunda a Domingo
ORDER BY
    CASE cal.dia_semana_num
        WHEN 1 THEN 1  -- Segunda
        WHEN 2 THEN 2  -- Terça
        WHEN 3 THEN 3  -- Quarta
        WHEN 4 THEN 4  -- Quinta
        WHEN 5 THEN 5  -- Sexta
        WHEN 6 THEN 6  -- Sábado
        WHEN 0 THEN 7  -- Domingo por último
    END;
"""

df_resultado = pd.read_sql(query, conn)

df_resultado['media_vendas'] = df_resultado['media_vendas'].apply(lambda x: f'R$ {x:,.2f}')
df_resultado['total_vendas'] = df_resultado['total_vendas'].apply(lambda x: f'R$ {x:,.0f}')

df_resultado[['dia_semana', 'total_dias', 'total_vendas', 'media_vendas']]

,dia_semana,total_dias,total_vendas,media_vendas
0,Segunda-feira,105,"R$ 363,839,460","R$ 3,465,137.71"
1,Terça-feira,105,"R$ 380,839,805","R$ 3,627,045.76"
2,Quarta-feira,104,"R$ 367,667,625","R$ 3,535,265.63"
3,Quinta-feira,104,"R$ 377,128,174","R$ 3,626,232.44"
4,Sexta-feira,104,"R$ 386,360,355","R$ 3,715,003.41"
5,Sábado,104,"R$ 385,896,217","R$ 3,710,540.55"
6,Domingo,105,"R$ 348,547,875","R$ 3,319,503.57"


Identificando o pior dia

In [4]:
query_pior = """
WITH vendas_diarias AS (
    SELECT DATE(sale_date) AS data, SUM(total) AS total_dia
    FROM vendas
    GROUP BY DATE(sale_date)
)
SELECT
    cal.dia_semana,
    ROUND(AVG(COALESCE(v.total_dia, 0)), 2) AS media_vendas
FROM dim_calendario cal
LEFT JOIN vendas_diarias v ON cal.data = v.data
GROUP BY cal.dia_semana_num, cal.dia_semana
ORDER BY media_vendas ASC
LIMIT 1;
"""

pior = pd.read_sql(query_pior, conn).iloc[0]
print(f'Pior dia da semana: {pior["dia_semana"]}')
print(f'Média de vendas:    R$ {pior["media_vendas"]:,.2f}')

conn.close()

Pior dia da semana: Domingo
Média de vendas:    R$ 3,319,503.57


---
### Querys SQLite

```bash
sqlite3 database/lighthouse.db
```

```sql
WITH vendas_diarias AS (
    SELECT DATE(sale_date) AS data, SUM(total) AS total_dia
    FROM vendas
    GROUP BY DATE(sale_date)
)
SELECT
    cal.dia_semana,
    COUNT(cal.data)                         AS total_dias,
    ROUND(AVG(COALESCE(v.total_dia, 0)), 2) AS media_vendas
FROM dim_calendario cal
LEFT JOIN vendas_diarias v ON cal.data = v.data
GROUP BY cal.dia_semana_num, cal.dia_semana
ORDER BY
    CASE cal.dia_semana_num
        WHEN 1 THEN 1 WHEN 2 THEN 2 WHEN 3 THEN 3
        WHEN 4 THEN 4 WHEN 5 THEN 5 WHEN 6 THEN 6 WHEN 0 THEN 7
    END;
```

### Questão 6.1
```sql
-- Criando o calendário com todos os dias do período
WITH RECURSIVE dim_calendario(data) AS (
    SELECT DATE(MIN(sale_date))
    FROM vendas

    UNION ALL

    SELECT DATE(data, '+1 day')
    FROM dim_calendario
    WHERE data < (SELECT DATE(MAX(sale_date)) FROM vendas)
),

-- Soma as vendas por dia
vendas_diarias AS (
    SELECT
        DATE(sale_date)  AS data,
        SUM(total)       AS total_dia
    FROM vendas
    GROUP BY DATE(sale_date)
)

-- LEFT JOIN + COALESCE + média por dia da semana
SELECT
    CASE strftime('%w', cal.data)
        WHEN '0' THEN 'Domingo'
        WHEN '1' THEN 'Segunda-feira'
        WHEN '2' THEN 'Terça-feira'
        WHEN '3' THEN 'Quarta-feira'
        WHEN '4' THEN 'Quinta-feira'
        WHEN '5' THEN 'Sexta-feira'
        WHEN '6' THEN 'Sábado'
    END                                          AS dia_semana,

    COUNT(cal.data)                              AS total_dias,
    SUM(COALESCE(v.total_dia, 0))                AS total_vendas,
    ROUND(AVG(COALESCE(v.total_dia, 0)), 2)      AS media_vendas

FROM dim_calendario cal
LEFT JOIN vendas_diarias v ON cal.data = v.data

GROUP BY strftime('%w', cal.data)

ORDER BY

    CASE strftime('%w', cal.data)
        WHEN '1' THEN 1
        WHEN '2' THEN 2
        WHEN '3' THEN 3
        WHEN '4' THEN 4
        WHEN '5' THEN 5
        WHEN '6' THEN 6
        WHEN '0' THEN 7
    END;
```

### Questão 6.2 — Validação
- Após considerar os dias zerados no cálculo: Qual é o Dia da Semana (ex: Domingo, Segunda...) que apresenta a menor média de vendas histórica, e qual é o valor dessa média arredondada para 2 casas decimais?

Domingo, com média de R$ 3.319.503,57.

### Questão 6.3 — Explicação
- Por que é necessário utilizar uma tabela de datas (calendário) em vez de agrupar diretamente a tabela de vendas? 
- O que aconteceria com a média de vendas se um dia da semana tivesse muitos dias sem nenhuma venda registrada?

A tabela de calendário é necessária porque a tabela de vendas só contém dias em que houve pelo menos uma transação — dias sem venda simplesmente não existem nela. Ao agrupar diretamente por dia_semana, o denominador da média conta apenas os dias com venda, ignorando os dias zerados e inflando o resultado. Com o LEFT JOIN partindo do calendário completo, todos os dias do período aparecem na query; o COALESCE transforma os NULL dos dias sem venda em 0, e esses zeros entram no cálculo da média reduzindo corretamente o valor. Se um dia da semana tivesse muitos dias sem venda e esses fossem ignorados, sua média apareceria artificialmente alta — podendo inclusive parecer o melhor dia da semana quando na prática é o pior, exatamente o erro que o estagiário cometeu com a Terça-feira.